# Tensors

A tensor is an n-dimensional array that can live on a GPU and can track gradients. It is the basic data type in PyTorch: inputs, outputs and model weights are all tensors.

Topics here follow the PyTorch basics tutorial on tensors: creating them, their attributes, common operations, and the link to NumPy.

In [ ]:
import numpy as np
import torch

## Creating tensors

From a Python list. The dtype is inferred, so a list of ints gives `int64`.

In [ ]:
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)

print(x_data)
print(x_data.dtype)

From a NumPy array.

In [ ]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

print(x_np)

From another tensor. The `_like` functions keep the shape of the argument and let you override the dtype.

In [ ]:
x_ones = torch.ones_like(x_data)
x_rand = torch.rand_like(x_data, dtype=torch.float)

print(x_ones)
print(x_rand)

From a shape tuple.

In [ ]:
shape = (2, 3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(rand_tensor)
print(ones_tensor)
print(zeros_tensor)

## Attributes

Shape, dtype and device are the three things to check first when something breaks.

In [ ]:
tensor = torch.rand(3, 4)

print(tensor.shape)
print(tensor.dtype)
print(tensor.device)

## Moving to a device

Tensors are created on the CPU and have to be moved. `.to()` returns a new tensor, it does not move in place, so the result must be assigned.

On this Mac the device is `mps`. Two tensors in one operation must sit on the same device.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

tensor = tensor.to(device)

print(device)
print(tensor.device)

## Indexing and slicing

Same rules as NumPy.

In [ ]:
tensor = torch.ones(4, 4)

print(tensor[0])
print(tensor[:, 0])
print(tensor[..., -1])

tensor[:, 1] = 0
print(tensor)

## Joining tensors

`torch.cat` joins along an existing dimension, so the shape grows along `dim`. Here 4 columns become 12.

In [ ]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)

print(t1)
print(t1.shape)

## Matrix multiplication

Three ways to write the same thing. `tensor.T` transposes, so `(4, 4) @ (4, 4)` gives `(4, 4)`.

In [ ]:
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)

y3 = torch.zeros_like(y1)
torch.matmul(tensor, tensor.T, out=y3)

print(y1)
print(y2)
print(y3)

## Element-wise product

Not the same as matrix multiplication. Every element is multiplied by the element in the same position.

In [ ]:
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.zeros_like(tensor)
torch.mul(tensor, tensor, out=z3)

print(z1)
print(z2)
print(z3)

## Single element tensors

An aggregation like `sum()` still returns a tensor. Use `.item()` to get a plain Python number, for example when logging a loss.

In [ ]:
agg = tensor.sum()
agg_item = agg.item()

print(agg)
print(agg_item)
print(type(agg_item))

## In-place operations

Methods ending with `_` change the tensor itself. They save memory but destroy the old value, which is why autograd does not always allow them.

In [ ]:
print(tensor)

tensor.add_(5)
print(tensor)

## Bridge with NumPy

A CPU tensor and the NumPy array made from it share the same memory. Changing one changes the other.

This works for CPU tensors only. On `mps` or `cuda` you have to call `.cpu()` first, otherwise `.numpy()` raises an error.

In [ ]:
t = torch.ones(5)
n = t.numpy()

print(t)
print(n)

t.add_(1)

print(t)
print(n)

The same holds in the other direction.

In [ ]:
n = np.ones(5)
t = torch.from_numpy(n)

np.add(n, 1, out=n)

print(t)
print(n)